# 🔴 Sensor Data Injector

Generates simulated IoT sensor readings and streams them into the **EH_SensorTelemetry** Eventhouse via Kusto inline ingestion.

**Modes:**
- **Batch** — Inject N historical readings in one shot
- **Continuous** — Stream readings every 2 seconds in real-time

Configure the mode and parameters in the **Configuration** cell below, then **Run All**.

## ⚙️ Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values before running
# ============================================================

# Mode: "batch" or "continuous"
MODE = "batch"

# Batch mode: number of readings to generate
BATCH_COUNT = 500

# Continuous mode: duration in seconds (0 = run until stopped)
DURATION_SECONDS = 300

# Streaming parameters
INTERVAL_SECONDS = 2       # Delay between batches
BATCH_SIZE = 60            # Sensors per batch
ANOMALY_PCT = 0.03         # 3% anomaly rate
ALERT_THRESHOLD_PCT = 0.05 # 5% of anomalies trigger alerts

# Resource IDs (from deployment)
QUERY_SERVICE_URI = "https://trd-qfhrr3s182t03md55n.z9.kusto.fabric.microsoft.com"
DATABASE_NAME = "EH_SensorTelemetry"
LAKEHOUSE_NAME = "LH_SensorReference"

## 🔐 Authentication & Sensor Data

In [ ]:
import requests
import math
import random
import time
import uuid
from datetime import datetime, timezone

# --- Auth: get Kusto token via Fabric notebook credentials ---
kusto_token = notebookutils.credentials.getToken(QUERY_SERVICE_URI)
print(f"\u2705 Kusto token acquired ({len(kusto_token)} chars)")

# --- Load sensor referential from Lakehouse ---
df_sensors = spark.sql(f"""
    SELECT * FROM {LAKEHOUSE_NAME}.dim_sensors
""")

sensors = []
for row in df_sensors.collect():
    sensors.append({
        "sensor_id":    row["sensor_id"],
        "sensor_type":  row["sensor_type"],
        "unit":         row["unit"],
        "zone_id":      row["zone_id"],
        "site_id":      row["site_id"],
        "min_normal":   float(row["min_normal"]),
        "max_normal":   float(row["max_normal"]),
        "min_critical": float(row["min_critical"]),
        "max_critical": float(row["max_critical"]),
    })

print(f"\u2705 Loaded {len(sensors)} sensors from Lakehouse")

## 🏭 Data Generation Functions

In [ ]:
def generate_reading(sensor, timestamp, anomaly_pct):
    """Generate a single sensor reading with optional anomaly."""
    is_anomaly = random.random() < anomaly_pct
    min_n, max_n = sensor["min_normal"], sensor["max_normal"]
    mid = (min_n + max_n) / 2
    spread = (max_n - min_n) / 2

    if is_anomaly:
        if random.random() < 0.5:
            value = random.uniform(sensor["min_critical"], min_n * 0.95)
        else:
            value = random.uniform(max_n * 1.05, sensor["max_critical"])
        quality = "Suspect"
    else:
        t = timestamp.timestamp()
        daily_phase = math.sin(2 * math.pi * (t % 86400) / 86400)
        noise = random.gauss(0, spread * 0.15)
        value = mid + daily_phase * spread * 0.4 + noise
        value = max(min_n * 0.98, min(max_n * 1.02, value))
        quality = "Good"

    return {
        "ReadingId":    str(uuid.uuid4())[:12],
        "SensorId":     sensor["sensor_id"],
        "ZoneId":       sensor["zone_id"],
        "SiteId":       sensor["site_id"],
        "Timestamp":    timestamp.strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
        "SensorType":   sensor["sensor_type"],
        "ReadingValue": round(value, 2),
        "Unit":         sensor["unit"],
        "IsAnomaly":    str(is_anomaly).lower(),
        "QualityFlag":  quality,
    }


def generate_alert(reading, sensor):
    """Generate an alert from an anomalous reading."""
    value = reading["ReadingValue"]
    if value < sensor["min_normal"]:
        alert_type = "BelowThreshold"
        threshold = sensor["min_normal"]
        severity = "Critical" if value < sensor["min_critical"] * 1.1 else "Warning"
        msg = f"{sensor['sensor_type']} too low: {value} {sensor['unit']} (min: {threshold})"
    else:
        alert_type = "AboveThreshold"
        threshold = sensor["max_normal"]
        severity = "Critical" if value > sensor["max_critical"] * 0.9 else "Warning"
        msg = f"{sensor['sensor_type']} too high: {value} {sensor['unit']} (max: {threshold})"

    return {
        "AlertId":        str(uuid.uuid4())[:12],
        "SensorId":       reading["SensorId"],
        "ZoneId":         reading["ZoneId"],
        "SiteId":         reading["SiteId"],
        "Timestamp":      reading["Timestamp"],
        "SensorType":     reading["SensorType"],
        "AlertType":      alert_type,
        "Severity":       severity,
        "ReadingValue":   reading["ReadingValue"],
        "ThresholdValue": round(threshold, 2),
        "Message":        msg,
    }

print("\u2705 Generation functions defined")

## 📤 Kusto Ingestion Functions

In [ ]:
def kusto_mgmt(command):
    """Execute a Kusto management command."""
    headers = {
        "Authorization": f"Bearer {kusto_token}",
        "Content-Type": "application/json; charset=utf-8",
    }
    body = {"db": DATABASE_NAME, "csl": command}
    resp = requests.post(
        f"{QUERY_SERVICE_URI}/v1/rest/mgmt",
        headers=headers, json=body, timeout=60
    )
    resp.raise_for_status()
    return resp.json()


def kusto_query(query):
    """Execute a Kusto query."""
    headers = {
        "Authorization": f"Bearer {kusto_token}",
        "Content-Type": "application/json; charset=utf-8",
    }
    body = {"db": DATABASE_NAME, "csl": query}
    resp = requests.post(
        f"{QUERY_SERVICE_URI}/v1/rest/query",
        headers=headers, json=body, timeout=60
    )
    resp.raise_for_status()
    return resp.json()


def ingest_batch(readings, alerts):
    """Ingest a batch of readings and alerts via inline ingestion."""
    if readings:
        lines = []
        for r in readings:
            lines.append(",".join([
                r["ReadingId"], r["SensorId"], r["ZoneId"], r["SiteId"],
                r["Timestamp"], r["SensorType"], str(r["ReadingValue"]),
                r["Unit"], r["IsAnomaly"], r["QualityFlag"],
            ]))
        for i in range(0, len(lines), 50):
            batch = lines[i:i + 50]
            inline = "\n".join(batch)
            kusto_mgmt(f".ingest inline into table SensorReading with (format='csv') <|\n{inline}")

    if alerts:
        lines = []
        for a in alerts:
            msg = a["Message"].replace('"', '""')
            lines.append(",".join([
                a["AlertId"], a["SensorId"], a["ZoneId"], a["SiteId"],
                a["Timestamp"], a["SensorType"], a["AlertType"],
                a["Severity"], str(a["ReadingValue"]),
                str(a["ThresholdValue"]), '"' + msg + '"',
            ]))
        for i in range(0, len(lines), 50):
            batch = lines[i:i + 50]
            inline = "\n".join(batch)
            kusto_mgmt(f".ingest inline into table SensorAlert with (format='csv') <|\n{inline}")

print("\u2705 Ingestion functions defined")

## 🔍 Pre-flight Check

Verify that the KQL tables exist and show current row counts.

In [ ]:
result = kusto_query("SensorReading | count")
reading_count = result["Tables"][0]["Rows"][0][0]

result2 = kusto_query("SensorAlert | count")
alert_count = result2["Tables"][0]["Rows"][0][0]

print(f"\U0001f4ca Current state:")
print(f"   SensorReading: {reading_count:,} rows")
print(f"   SensorAlert:   {alert_count:,} rows")
print(f"   Sensors loaded: {len(sensors)}")
print(f"   Mode: {MODE}")
if MODE == "batch":
    print(f"   Batch count: {BATCH_COUNT:,}")
else:
    print(f"   Duration: {DURATION_SECONDS}s (0=infinite)")
    print(f"   Interval: {INTERVAL_SECONDS}s, Batch size: {BATCH_SIZE}")
print(f"\n\u2705 Pre-flight OK \u2014 ready to inject")

## 🚀 Run Injection

Execute the injection based on the selected mode.

In [ ]:
def run_batch_mode(count):
    """Inject a fixed number of historical readings."""
    readings = []
    alerts = []
    now = datetime.now(timezone.utc)

    print(f"\U0001f4e6 Generating {count:,} readings...")
    for i in range(count):
        sensor = random.choice(sensors)
        ts = datetime.fromtimestamp(
            now.timestamp() - (count - i) * 2, tz=timezone.utc
        )
        reading = generate_reading(sensor, ts, ANOMALY_PCT)
        readings.append(reading)
        if reading["IsAnomaly"] == "true":
            alerts.append(generate_alert(reading, sensor))

    print(f"\U0001f4e4 Ingesting {len(readings):,} readings, {len(alerts):,} alerts...")
    ingest_batch(readings, alerts)
    print(f"\u2705 Batch injection complete!")
    return len(readings), len(alerts)


def run_continuous_mode(duration):
    """Stream readings in real time."""
    total_readings = 0
    total_alerts = 0
    start_time = time.time()

    print(f"\U0001f534 LIVE \u2014 Injecting every {INTERVAL_SECONDS}s "
          f"({len(sensors)} sensors, batch size {BATCH_SIZE})")
    if duration:
        print(f"   Will stop after {duration}s")
    print(f"   Stop the cell to interrupt\n")

    while True:
        if duration and (time.time() - start_time) > duration:
            break

        now = datetime.now(timezone.utc)
        readings = []
        alerts = []

        batch_sensors = random.sample(
            sensors, min(BATCH_SIZE, len(sensors))
        )

        for sensor in batch_sensors:
            reading = generate_reading(sensor, now, ANOMALY_PCT)
            readings.append(reading)
            if reading["IsAnomaly"] == "true" and random.random() < ALERT_THRESHOLD_PCT / ANOMALY_PCT:
                alerts.append(generate_alert(reading, sensor))

        ingest_batch(readings, alerts)
        total_readings += len(readings)
        total_alerts += len(alerts)

        elapsed = int(time.time() - start_time)
        print(f"\r  \U0001f4ca {total_readings:,} readings, {total_alerts:,} alerts "
              f"({elapsed}s elapsed)", end="", flush=True)

        time.sleep(INTERVAL_SECONDS)

    elapsed = int(time.time() - start_time)
    print(f"\n\u2705 Injected {total_readings:,} readings and {total_alerts:,} alerts "
          f"in {elapsed}s")
    return total_readings, total_alerts


# --- Execute ---
if MODE == "batch":
    r_count, a_count = run_batch_mode(BATCH_COUNT)
else:
    r_count, a_count = run_continuous_mode(DURATION_SECONDS)

## 📊 Post-Injection Verification

Query the Eventhouse to confirm data landed.

In [ ]:
# Row counts
result = kusto_query("SensorReading | count")
new_reading_count = result["Tables"][0]["Rows"][0][0]

result2 = kusto_query("SensorAlert | count")
new_alert_count = result2["Tables"][0]["Rows"][0][0]

print(f"\U0001f4ca Final row counts:")
print(f"   SensorReading: {new_reading_count:,} rows")
print(f"   SensorAlert:   {new_alert_count:,} rows")

# Latest 5 readings
result3 = kusto_query("SensorReading | top 5 by Timestamp desc")
cols = [c["ColumnName"] for c in result3["Tables"][0]["Columns"]]
print(f"\n\U0001f553 Latest 5 readings:")
print(f"   {' | '.join(cols[:6])}")
print(f"   {'\u2014' * 80}")
for row in result3["Tables"][0]["Rows"]:
    print(f"   {' | '.join(str(v) for v in row[:6])}")

# Anomaly breakdown
result4 = kusto_query("""
    SensorReading
    | summarize Total=count(), Anomalies=countif(IsAnomaly == true) by SensorType
    | extend AnomalyPct = round(100.0 * Anomalies / Total, 1)
    | order by AnomalyPct desc
""")
print(f"\n\U0001f50d Anomaly breakdown by sensor type:")
for row in result4["Tables"][0]["Rows"]:
    print(f"   {row[0]:12s}  Total: {row[1]:>6,}  Anomalies: {row[2]:>4}  ({row[3]}%)")